In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import time
import numpy as np
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import optuna
from optuna.pruners import MedianPruner
import xgboost as xgb

In [ ]:
X = pd.read_csv("train_features_cleaned.csv")
Y = pd.read_csv("train_target_corr.csv").squeeze()
X_test_final = pd.read_csv("test_features_cleaned.csv")
Y=Y['change_type']
print(X.shape, Y.shape, X_test_final.shape)


In [ ]:
# =========================================================
# XGBOOST MULTICLASS + OPTUNA + STRATIFIED CV (GPU)
# + WEIGHTS (imbalance)
# + THRESHOLD TUNING (post-processing) for classes 4 and 5
#
# Idea:
#   1) Optuna tunes XGBoost params using CV macro-F1 computed AFTER thresholding.
#   2) Inside each trial, we also tune thresholds (t4, t5) on out-of-fold proba.
#   3) Train final model on all data, tune (t4,t5) once on OOF proba, predict test.
# =========================================================


print("Script started")
print("xgboost version:", xgb.__version__)

# =========================================================
# 0) CAST float32 + keep test index
# =========================================================
test_index = X_test_final.index if hasattr(X_test_final, "index") else None

def to_float32(A):
    if hasattr(A, "to_numpy"):
        return A.to_numpy(dtype=np.float32, copy=False)
    return np.asarray(A, dtype=np.float32)

X_train = to_float32(X)
X_test  = to_float32(X_test_final)
y = np.asarray(Y).astype(np.int32)

NUM_CLASS = 6
print("Classes présentes:", np.unique(y))
print("Distribution:", Counter(y))

# =========================================================
# 1) Weights (inverse freq, capped)
# =========================================================
def inverse_freq_weights(y_vec, cap=50.0):
    counts = Counter(y_vec)
    n = len(y_vec)
    k = len(counts)
    weights = {c: n / (k * cnt) for c, cnt in counts.items()}
    w = np.array([weights[int(v)] for v in y_vec], dtype=np.float32)
    return np.minimum(w, cap).astype(np.float32)

# =========================================================
# 2) Thresholded prediction (STRICT post-processing)
# =========================================================
def apply_thresholds_strict(proba, t4, t5):
    """
    proba: (n,6) softprob
    Règle:
      - classes 4 et 5 ne sont autorisées que si P(4) > t4 / P(5) > t5
      - sinon, on choisit la meilleure classe parmi {0,1,2,3}
      - si 4 et 5 passent tous les deux, on prend celle des deux avec la plus grande proba
        (priorité implicite à la plus grande proba)
    """
    proba = np.asarray(proba)
    n, k = proba.shape
    assert k == 6, "On attend 6 classes (0..5)."

    p4 = proba[:, 4]
    p5 = proba[:, 5]

    ok4 = p4 > t4
    ok5 = p5 > t5

    # meilleur parmi 0..3
    pred_0_3 = np.argmax(proba[:, :4], axis=1).astype(np.int32)

    pred = pred_0_3.copy()

    # si seulement 4 est autorisée
    pred[ok4 & ~ok5] = 4

    # si seulement 5 est autorisée
    pred[ok5 & ~ok4] = 5

    # si 4 et 5 sont toutes les deux autorisées -> choisir la plus probable entre 4 et 5
    both = ok4 & ok5
    pred[both] = np.where(p5[both] >= p4[both], 5, 4).astype(np.int32)

    return pred

def tune_thresholds_grid(proba, y_true,
                         t4_min=0.20, t4_max=0.99, t4_steps=33,
                         t5_min=0.20, t5_max=0.99, t5_steps=33):
    """
    Grid-search thresholds to maximize macro-F1 on given (proba, y_true).
    (utilise apply_thresholds_strict)
    """
    best_f1 = -1.0
    best_t4 = 0.5
    best_t5 = 0.7

    t4_grid = np.linspace(t4_min, t4_max, t4_steps)
    t5_grid = np.linspace(t5_min, t5_max, t5_steps)

    for t4 in t4_grid:
        for t5 in t5_grid:
            pred = apply_thresholds_strict(proba, t4=float(t4), t5=float(t5))
            score = f1_score(y_true, pred, average="macro", zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_t4 = float(t4)
                best_t5 = float(t5)

    return best_t4, best_t5, float(best_f1)

# =========================================================
# 3) CV + Optuna Objective (macro-F1 AFTER threshold tuning)
# =========================================================
N_SPLITS = 3
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

def objective(trial):
    start = time.time()
    print(f"\nTrial {trial.number} started")

    # --- you can also tune weight cap (optional, safe range)
    w_cap = trial.suggest_float("w_cap", 5.0, 60.0, log=True)

    params = {
        "objective": "multi:softprob",
        "num_class": NUM_CLASS,
        "eval_metric": "mlogloss",

        "tree_method": "hist",
        "device": "cuda",

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 30, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 15.0),
        "alpha": trial.suggest_float("alpha", 1e-8, 20.0, log=True),
        "lambda": trial.suggest_float("lambda", 1e-3, 200.0, log=True),
        "max_delta_step": trial.suggest_float("max_delta_step", 0.0, 10.0),

        "verbosity": 0,
    }

    # ---- Collect out-of-fold probabilities for threshold tuning
    oof_proba = np.zeros((len(y), NUM_CLASS), dtype=np.float32)

    best_iters = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y), start=1):
        X_tr, X_va = X_train[tr_idx], X_train[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        w_tr = inverse_freq_weights(y_tr, cap=w_cap)

        dtr = xgb.QuantileDMatrix(X_tr, label=y_tr, weight=w_tr)
        dva = xgb.QuantileDMatrix(X_va, label=y_va, ref=dtr)

        booster = xgb.train(
            params=params,
            dtrain=dtr,
            num_boost_round=5000,
            evals=[(dva, f"val_fold{fold}")],
            early_stopping_rounds=150,
            verbose_eval=0
        )

        proba_va = booster.predict(dva, iteration_range=(0, booster.best_iteration + 1))
        oof_proba[va_idx] = proba_va
        best_iters.append(booster.best_iteration)

    mean_best_iter = int(np.round(np.mean(best_iters)))

    # ---- Tune thresholds on OOF predictions (no leakage)
    t4, t5, tuned_f1 = tune_thresholds_grid(
        oof_proba, y,
        t4_min=0.10, t4_max=0.95, t4_steps=35,
        t5_min=0.10, t5_max=0.98, t5_steps=35
    )

    trial.set_user_attr("mean_best_iter", mean_best_iter)
    trial.set_user_attr("best_t4", t4)
    trial.set_user_attr("best_t5", t5)

    elapsed = time.time() - start
    print(f"Trial {trial.number} done | OOF tuned Macro-F1={tuned_f1:.5f} | t4={t4:.3f} t5={t5:.3f} | iter≈{mean_best_iter} | time={elapsed:.1f}s")

    return tuned_f1

# =========================================================
# 4) OPTUNA
# =========================================================
print("\n Starting Optuna tuning (CV + threshold tuning)...\n")

study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_startup_trials=10))
study.optimize(objective, n_trials=80, show_progress_bar=True)

best_params = study.best_params
best_iter   = int(study.best_trial.user_attrs.get("mean_best_iter", 800))
best_t4     = float(study.best_trial.user_attrs.get("best_t4", 0.5))
best_t5     = float(study.best_trial.user_attrs.get("best_t5", 0.7))

print("\n BEST tuned OOF macro-F1:", study.best_value)
print(" BEST PARAMS:", best_params)
print(" BEST mean_best_iter:", best_iter)
print(" BEST thresholds:", {"t4": best_t4, "t5": best_t5})

# =========================================================
# 5) TRAIN FINAL MODEL on ALL DATA + WEIGHTS
# =========================================================
print("\n Training final model on ALL data...")

final_params = {
    "objective": "multi:softprob",
    "num_class": NUM_CLASS,
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "device": "cuda",

    "learning_rate": best_params["learning_rate"],
    "max_depth": best_params["max_depth"],
    "min_child_weight": best_params["min_child_weight"],
    "subsample": best_params["subsample"],
    "colsample_bytree": best_params["colsample_bytree"],
    "gamma": best_params["gamma"],
    "alpha": best_params["alpha"],
    "lambda": best_params["lambda"],
    "max_delta_step": best_params["max_delta_step"],

    "verbosity": 1,
}

w_all = inverse_freq_weights(y, cap=best_params["w_cap"])
dall = xgb.QuantileDMatrix(X_train, label=y, weight=w_all)

final_booster = xgb.train(
    params=final_params,
    dtrain=dall,
    num_boost_round=best_iter,
    evals=[(dall, "train")],
    verbose_eval=25
)

print(" Final model trained")

# =========================================================
# 6) PREDICT TEST with tuned thresholds + SAVE
# =========================================================
print("\n Predicting on X_test_final...")

dtest = xgb.QuantileDMatrix(X_test, ref=dall)
proba_test = final_booster.predict(dtest)

final_pred = apply_thresholds_strict(proba_test, t4=best_t4, t5=best_t5)
print(" Prediction finished | Pred distribution:", Counter(final_pred))

import pandas as pd
index_values = test_index if test_index is not None else np.arange(len(final_pred))

df_pred = pd.DataFrame({
    "index": index_values,
    "prediction": final_pred
})
df_pred.to_csv("predictions5.csv", index=False)

print(" Saved → predictions5.csv")
